# Lab M14 — Building and Evaluating a Baseline Model (Solution)

> **Goal:** Build and evaluate **baseline predictive models** using Scikit-Learn, comparing **regression** and **classification** approaches.  
> **Dataset:** `validated_dataset_m14.csv`

---

## How to use this notebook
- Run cells top to bottom.
- Read the **markdown instructions** carefully.
- Fill in the **TODO** sections (you'll see them clearly marked).
- Save outputs to the `outputs/` folder.

✅ By the end, you will have:
- A trained regression model (Linear Regression)  
- A trained classification model (Logistic Regression)  
- Evaluation metrics (RMSE, MAE, Accuracy) saved to `outputs/model_metrics.csv`  
- Visualizations comparing predictions to actual values

## 📚 Connection to Module 13

In Module 13, you learned to **forecast future values** using time-based patterns (naive, mean, rolling average forecasts).

In this module, you'll learn to **predict outcomes based on features** using machine learning:

| Module 13 (Forecasting) | Module 14 (Machine Learning) |
|------------------------|-----------------------------|
| Predict future values | Predict outcomes from features |
| Uses temporal patterns | Uses feature relationships |
| Naive, Mean, Rolling forecasts | Linear/Logistic Regression |
| MAE, RMSE for evaluation | MAE, RMSE, Accuracy for evaluation |

Both approaches are valuable — the choice depends on your prediction task.

In [ ]:
# 1) Setup
# Import all required libraries

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Scikit-Learn imports
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score

%matplotlib inline

# Create output folders (safe to re-run)
Path("outputs/figures").mkdir(parents=True, exist_ok=True)

print("✅ Setup complete")

## 2) Load and Explore the Dataset

We'll use the same validated dataset from Module 13, but now we'll approach it as a **machine learning problem** rather than a time series forecasting problem.

You will:
1. Load the CSV from `data/validated_dataset_m14.csv`
2. Explore the available features
3. Understand what we can predict

In [ ]:
# 2) Load dataset (project-root anchored so it works from /notebooks)
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
data_path = PROJECT_ROOT / "data" / "validated_dataset_m14.csv"

df = pd.read_csv(data_path)

print(f"Loaded: {data_path}")
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("Columns:")
print(df.columns.tolist())


In [ ]:
# Explore the first few rows
df.head()

In [ ]:
# Check data types and missing values
df.info()

In [ ]:
# Summary statistics for numeric columns
df.describe()

## 3) Understand the Prediction Tasks

We'll work on **two prediction tasks**:

### Task 1: Regression
**Predict `revenue_usd`** (continuous value) based on features like price, discount, marketing spend, etc.

### Task 2: Classification  
**Predict `high_revenue`** (binary: Yes/No) — we'll create this variable based on whether revenue is above the median.

✅ **Solution note:** The target variable `revenue_usd` has a wide range and is right-skewed due to some high-value products. This is typical of sales data.

In [ ]:
# Plot the distribution of revenue_usd
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.hist(df['revenue_usd'], bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Revenue (USD)')
plt.ylabel('Frequency')
plt.title('Distribution of Revenue')

plt.subplot(1, 2, 2)
plt.boxplot(df['revenue_usd'])
plt.ylabel('Revenue (USD)')
plt.title('Revenue Box Plot')

plt.tight_layout()
plt.savefig('outputs/figures/revenue_distribution.png', dpi=150)
plt.show()

print(f"\n📊 Revenue Statistics:")
print(f"   Mean:   ${df['revenue_usd'].mean():,.2f}")
print(f"   Median: ${df['revenue_usd'].median():,.2f}")
print(f"   Std:    ${df['revenue_usd'].std():,.2f}")

### Create Classification Target

For the classification task, we'll create a binary variable `high_revenue`:
- **1** if revenue is above the median
- **0** if revenue is at or below the median

✅ **Solution:** Using the median as threshold ensures balanced classes (approximately 50/50 split).

In [ ]:
# Create binary classification target
median_revenue = df['revenue_usd'].median()
df['high_revenue'] = (df['revenue_usd'] > median_revenue).astype(int)

print(f"📊 Median revenue threshold: ${median_revenue:,.2f}")
print(f"\n📋 Class distribution:")
print(df['high_revenue'].value_counts())
print(f"\n📊 Class balance: {df['high_revenue'].mean():.1%} high revenue")

## 4) Select Features and Target Variables

For machine learning, we need to separate:
- **Features (X)**: Input variables used to make predictions
- **Target (y)**: Output variable we want to predict

✅ **Solution:** We select numeric features that logically influence revenue:
- Higher price → potentially higher revenue per unit
- Higher discount → may increase volume but reduce per-unit revenue
- Marketing spend → should drive traffic and sales
- Web sessions → more visitors may mean more sales
- Units sold → directly affects total revenue

In [ ]:
# Define the feature columns
feature_cols = ['price_usd', 'discount_pct', 'marketing_spend_usd', 'web_sessions', 'units_sold']

# Create feature matrix X
X = df[feature_cols].copy()

# Create target variables
y_regression = df['revenue_usd'].copy()      # For regression
y_classification = df['high_revenue'].copy()  # For classification

print(f"✅ Features shape: {X.shape}")
print(f"✅ Regression target shape: {y_regression.shape}")
print(f"✅ Classification target shape: {y_classification.shape}")

In [ ]:
# Check for missing values in features
print("📋 Missing values per feature:")
print(X.isnull().sum())

# If there are missing values, handle them
if X.isnull().sum().sum() > 0:
    print("\n⚠️ Handling missing values by dropping rows...")
    mask = X.notnull().all(axis=1)
    X = X[mask]
    y_regression = y_regression[mask]
    y_classification = y_classification[mask]
    print(f"✅ New shape: {X.shape}")
else:
    print("\n✅ No missing values found!")

## 5) Split Data into Training and Test Sets

**Critical concept:** We must evaluate our model on data it has never seen during training.

✅ **Solution:** 
- 80/20 split is standard for datasets of this size
- `random_state=42` ensures reproducibility
- For classification, `stratify` maintains class balance in both sets

In [ ]:
# Split data for REGRESSION task
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X, 
    y_regression, 
    test_size=0.2, 
    random_state=42
)

print("📊 Regression Split:")
print(f"   Training set: {X_train_reg.shape[0]} samples")
print(f"   Test set:     {X_test_reg.shape[0]} samples")

In [ ]:
# Split data for CLASSIFICATION task
# Note: We use stratify to maintain class balance in both sets
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X, 
    y_classification, 
    test_size=0.2, 
    random_state=42,
    stratify=y_classification  # Maintains class proportions
)

print("📊 Classification Split:")
print(f"   Training set: {X_train_clf.shape[0]} samples")
print(f"   Test set:     {X_test_clf.shape[0]} samples")
print(f"\n📋 Class distribution in test set:")
print(y_test_clf.value_counts())

## 6) Train Baseline Regression Model

**Linear Regression** assumes a linear relationship between features and target.

✅ **Solution:** The model learns coefficients for each feature that minimize prediction error.

In [ ]:
# Train Linear Regression model

# Step 1: Create model instance
reg_model = LinearRegression()

# Step 2: Fit on training data
reg_model.fit(X_train_reg, y_train_reg)

# Step 3: Predict on test data
y_pred_reg = reg_model.predict(X_test_reg)

print("✅ Linear Regression model trained!")
print(f"\n📊 Model coefficients:")
for feature, coef in zip(feature_cols, reg_model.coef_):
    print(f"   {feature}: {coef:.4f}")
print(f"   Intercept: {reg_model.intercept_:.4f}")

### Evaluate Regression Model

✅ **Solution interpretation:** 
- The `units_sold` coefficient is high because more units directly means more revenue
- The `price_usd` coefficient shows revenue increases with price (for same units)
- A negative `discount_pct` coefficient would indicate discounts reduce revenue per unit

In [ ]:
# Calculate regression metrics

# MAE
mae_reg = mean_absolute_error(y_test_reg, y_pred_reg)

# RMSE
rmse_reg = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))

print("📊 Regression Model Performance:")
print(f"   MAE:  ${mae_reg:,.2f}")
print(f"   RMSE: ${rmse_reg:,.2f}")
print(f"\n📝 Interpretation:")
print(f"   On average, predictions are off by ${mae_reg:,.2f}")
print(f"   Typical prediction error magnitude: ${rmse_reg:,.2f}")

In [ ]:
# Visualize: Actual vs Predicted
plt.figure(figsize=(10, 5))

# Scatter plot
plt.subplot(1, 2, 1)
plt.scatter(y_test_reg, y_pred_reg, alpha=0.5)
plt.plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], 'r--', label='Perfect prediction')
plt.xlabel('Actual Revenue (USD)')
plt.ylabel('Predicted Revenue (USD)')
plt.title('Actual vs Predicted Revenue')
plt.legend()

# Residuals plot
plt.subplot(1, 2, 2)
residuals = y_test_reg - y_pred_reg
plt.hist(residuals, bins=30, edgecolor='black', alpha=0.7)
plt.xlabel('Prediction Error (USD)')
plt.ylabel('Frequency')
plt.title('Distribution of Prediction Errors')
plt.axvline(x=0, color='r', linestyle='--', label='Zero error')
plt.legend()

plt.tight_layout()
plt.savefig('outputs/figures/regression_results.png', dpi=150)
plt.show()

print("✅ Saved: outputs/figures/regression_results.png")

## 7) Train Baseline Classification Model

✅ **Solution:** Logistic Regression is a great baseline for binary classification — simple, interpretable, and often surprisingly effective.

In [ ]:
# Train Logistic Regression model

# Step 1: Create model instance
clf_model = LogisticRegression(max_iter=1000, random_state=42)

# Step 2: Fit on training data
clf_model.fit(X_train_clf, y_train_clf)

# Step 3: Predict on test data
y_pred_clf = clf_model.predict(X_test_clf)

print("✅ Logistic Regression model trained!")

### Evaluate Classification Model

✅ **Solution:** With balanced classes, accuracy is a meaningful metric. An accuracy above 50% indicates the model learned something useful.

In [ ]:
# Calculate classification metrics

# Accuracy
accuracy_clf = accuracy_score(y_test_clf, y_pred_clf)

print("📊 Classification Model Performance:")
print(f"   Accuracy: {accuracy_clf:.2%}")
print(f"\n📝 Interpretation:")
print(f"   The model correctly predicts high/low revenue {accuracy_clf:.1%} of the time")

In [ ]:
# Create a simple confusion matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_clf, y_pred_clf)

plt.figure(figsize=(6, 5))
plt.imshow(cm, interpolation='nearest', cmap='Blues')
plt.title('Confusion Matrix')
plt.colorbar()

classes = ['Low Revenue', 'High Revenue']
tick_marks = [0, 1]
plt.xticks(tick_marks, classes)
plt.yticks(tick_marks, classes)

# Add text annotations
for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=20)

plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('outputs/figures/confusion_matrix.png', dpi=150)
plt.show()

print("✅ Saved: outputs/figures/confusion_matrix.png")

## 8) Save Model Metrics

In [ ]:
# Create and save metrics DataFrame
from pathlib import Path
import pandas as pd

metrics_data = [
    {
        "model": "Linear Regression",
        "task": "Regression",
        "target": "revenue_usd",
        "MAE": mae_reg,
        "RMSE": rmse_reg,
        "Accuracy": None,
    },
    {
        "model": "Logistic Regression",
        "task": "Classification",
        "target": "high_revenue",
        "MAE": None,
        "RMSE": None,
        "Accuracy": accuracy_clf,
    },
]

metrics_df = pd.DataFrame(metrics_data)

PROJECT_ROOT = Path.cwd().parent
OUT_DIR = PROJECT_ROOT / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

out_path = OUT_DIR / "model_metrics.csv"
metrics_df.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
display(metrics_df)


## 9) Compare with Module 13 Forecasting

| Aspect | M13: Time Series Forecasting | M14: Machine Learning |
|--------|------------------------------|----------------------|
| **Goal** | Predict future values | Predict outcomes from features |
| **Input** | Historical time series | Feature variables |
| **Models** | Naive, Mean, Rolling Average | Linear/Logistic Regression |
| **Data Split** | Chronological (train before test) | Random split |
| **Metrics** | MAE, RMSE | MAE, RMSE, Accuracy |

**Key insight:** Time series forecasting predicts *when*, ML predicts *what* based on *features*.

## 10) Interpret Results

✅ **Solution interpretation:**

### Solution Interpretation

- **Regression model performance:**  
  The Linear Regression model predicts revenue with reasonable accuracy. The MAE tells us the average prediction error in dollars, while RMSE being higher suggests some larger errors (likely on high-revenue transactions). The model captures the strong relationship between units_sold and revenue.

- **Classification model performance:**  
  The Logistic Regression model achieves accuracy above 50% (random guessing), indicating it learned meaningful patterns. The confusion matrix shows performance on both classes.

- **Limitations / risks:**  
  - Linear models assume linear relationships, which may not hold for all features
  - High-value outliers (luxury products) may skew predictions
  - The models don't account for temporal patterns (seasonality, trends)
  - Features like `units_sold` may be "leaky" — we often don't know units sold before making a prediction

- **What you would try next:**  
  - Remove `units_sold` to create a more realistic prediction scenario
  - Try tree-based models (Random Forest, Gradient Boosting)
  - Engineer new features (e.g., price × discount interaction)
  - Add categorical features (category, channel, state)
  - Use cross-validation for more robust evaluation

## AI Reflection Prompt

Before finalizing your submission, ask an AI assistant:

> **"What are the key differences between time series forecasting and supervised machine learning, and when should each approach be used?"**

### Solution AI Reflection

**Key differences:**
- **Time series forecasting** uses historical patterns (trends, seasonality) to predict future values. The order of data matters.
- **Supervised ML** uses feature-target relationships to predict outcomes. The order typically doesn't matter.

**When to use each:**
- Use **time series** when predicting future values based on temporal patterns (stock prices, demand forecasting, weather)
- Use **supervised ML** when predicting outcomes based on feature relationships (customer churn, fraud detection, price prediction)

**Connection to this lab:**
- In M13, we forecasted daily revenue based on past values
- In M14, we predicted revenue based on features like price, marketing spend, etc.
- Both approaches are valid — the choice depends on the question being asked

## ✅ Lab Summary

In this lab, you:

1. **Loaded and explored** the dataset
2. **Prepared features and targets** for both regression and classification
3. **Split data** into training and test sets (80/20)
4. **Trained baseline models** using Scikit-Learn
5. **Evaluated performance** using MAE, RMSE, and Accuracy
6. **Visualized predictions** and errors
7. **Saved metrics** to CSV
8. **Interpreted results** and identified limitations

**Key takeaways:**
- Always split data before training
- Baseline models provide a performance benchmark
- Different tasks require different metrics
- Interpretation is as important as computation